In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error



In [ ]:

# TASK 1: Load Ahmedabad Weather Data & Engineer Lag + Rolling Features

df_temp = pd.read_csv('ahmedabad_temperature.csv')

# Ensure Date column is in datetime format and establish it as the index
df_temp['Date'] = pd.to_datetime(df_temp['Date'])
df_temp.set_index('Date', inplace=True)

# 1. Create a 3-day historical lag feature
df_temp['Lag_3'] = df_temp['Temperature'].shift(3)

# 2. Create a 7-day rolling window moving average feature
df_temp['Rolling_7_Mean'] = df_temp['Temperature'].rolling(window=7).mean()

print(df_temp[['Temperature', 'Lag_3', 'Rolling_7_Mean']].dropna().head())
print("\n" + "="*60 + "\n")




In [ ]:

# TASK 2: Extract Datetime Attributes from Flipkart-Style Sales Data

df_sales = pd.read_csv('flipkart_sales_mock.csv')

df_sales['order_date'] = pd.to_datetime(df_sales['order_date'])

# Extracting temporal components
df_sales['day_of_week'] = df_sales['order_date'].dt.day_name()
df_sales['month'] = df_sales['order_date'].dt.month_name()
# Binary indicator: 1 for Saturday/Sunday, 0 for weekdays
df_sales['is_weekend'] = df_sales['order_date'].dt.dayofweek.isin([5, 6]).astype(int)

print(df_sales.head())
print("\n" + "="*60 + "\n")



In [ ]:


# TASK 3: Train Random Forest Regressor & Predict Next Value
# Drop NaN rows created by the rolling/shifting transformations
df_ml = df_temp.dropna().copy()

X = df_ml[['Lag_3', 'Rolling_7_Mean']]
y = df_ml['Temperature']

# Chronological Sequential Split (80% Train, 20% Test) to prevent data leakage
split_idx = int(len(df_ml) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Initialize and train the Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Calculate feature values for the next immediate unobserved date
# To forecast tomorrow, we need tomorrow's 3-day lag (which is 2 days ago from today) 
# and tomorrow's 7-day rolling mean (which includes today's temperature value)
next_date_features = pd.DataFrame([{
    'Lag_3': df_temp['Temperature'].iloc[-2], 
    'Rolling_7_Mean': df_temp['Temperature'].tail(7).mean()
}])

next_day_forecast = rf_model.predict(next_date_features)[0]
print(f"Predicted Temperature for the next chronological date: {next_day_forecast:.2f}°C")
print("\n" + "="*60 + "\n")



In [ ]:


# TASK 4: Traditional Loss Metrics Evaluation
y_pred = rf_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Custom Mean Absolute Percentage Error (MAPE) implementation
def calculate_mape(actual, predicted):
    return np.mean(np.abs((actual - predicted) / actual)) * 100

mape = calculate_mape(y_test, y_pred)

print(f"Mean Absolute Error (MAE)        : {mae:.4f}°C")
print(f"Root Mean Squared Error (RMSE)    : {rmse:.4f}°C")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.4f}%")
print("\n" + "="*60 + "\n")



In [ ]:


# TASK 5: AI-Refactored SMAPE Code & Structural Learning

def calculate_smape(actual, predicted):
    """
    Calculates Symmetric Mean Absolute Percentage Error (SMAPE)
    """
    actual = np.array(actual)
    predicted = np.array(predicted)
    
    # Avoid zero-division errors by setting an epsilon edge constraint 
    denominator = (np.abs(actual) + np.abs(predicted)) / 2
    
    # Mask out cases where both actual and predicted are exactly zero
    nonzero_mask = denominator != 0
    
    return np.mean(np.abs(predicted[nonzero_mask] - actual[nonzero_mask]) / denominator[nonzero_mask]) * 100

smape_score = calculate_smape(y_test, y_pred)
print(f"Symmetric Mean Absolute Percentage Error (SMAPE): {smape_score:.4f}%")